# Session 15: Automation and Network Programmability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-15-automation-and-programmability.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-15_automation.py`

In [1]:
"""Session 15 - Network automation with JSON, templates and configuration drift checks.

1. Load the "source of truth" (session-15_inventory.json) and validate it.
2. Generate each switch's configuration from one template (no copy-paste errors).
3. Compare an intended config with what is running and report drift.
4. Convert data between JSON and Python to show why APIs use JSON.
Run:  python session-15_automation.py
"""
import difflib
import ipaddress
import json


def load_text(name):
    """Read a data file next to this script; on Colab, fetch it from the course repo."""
    try:
        return open(name).read()
    except FileNotFoundError:
        import urllib.request
        url = ("https://raw.githubusercontent.com/Hamnakhan123/Cybersecurity-and-Digital-Forensics/"
               "main/term-1/04-configuring-network-infrastructure/code/" + name)
        return urllib.request.urlopen(url).read().decode()


INV = json.loads(load_text("session-15_inventory.json"))


def validate(inv):
    errors = []
    ids = [v["id"] for v in inv["vlans"]]
    if len(ids) != len(set(ids)):
        errors.append("duplicate VLAN id")
    nets = [ipaddress.ip_network(v["subnet"]) for v in inv["vlans"]]
    for i, a in enumerate(nets):
        for b in nets[i + 1:]:
            if a.overlaps(b):
                errors.append(f"{a} overlaps {b}")
    for sw in inv["switches"]:
        for vid in sw["access"]:
            if int(vid) not in ids:
                errors.append(f"{sw['hostname']} uses undefined VLAN {vid}")
    return errors


def render(inv, sw):
    lines = [f"hostname {sw['hostname']}", f"ntp server {inv['ntp']}", f"logging host {inv['syslog']}"]
    for v in inv["vlans"]:
        lines += [f"vlan {v['id']}", f" name {v['name']}"]
    for vid, ports in sw["access"].items():
        prefix, rng = ports.split("/")
        lines += [f"interface range {prefix}/{rng.replace('-', ' - ')}",
                  " switchport mode access", f" switchport access vlan {vid}",
                  " switchport nonegotiate", " spanning-tree portfast"]
    lines += ["interface vlan 99", f" ip address {sw['mgmt_ip']} 255.255.255.0"]
    return lines


if __name__ == "__main__":
    print(f"Source of truth for {INV['site']}: {len(INV['vlans'])} VLANs, {len(INV['switches'])} switches")
    print("Validation:", validate(INV) or "OK")

    configs = {sw["hostname"]: render(INV, sw) for sw in INV["switches"]}
    print("\nGenerated config for ACC-SW1:")
    print("\n".join("  " + l for l in configs["ACC-SW1"]))

    running = configs["ACC-SW2"].copy()
    running[1] = "ntp server 10.0.0.1"                         # someone changed it by hand
    running.insert(5, " name TEMP-TEST")                       # and added an untracked name
    print("\nDrift check on ACC-SW2 (intended vs running):")
    diff = list(difflib.unified_diff(configs["ACC-SW2"], running, "intended", "running", lineterm="", n=0))
    print("\n".join("  " + d for d in diff))

    bad = json.loads(json.dumps(INV))
    bad["vlans"].append({"id": 30, "name": "LAB", "subnet": "192.168.21.0/24"})
    bad["switches"][1]["access"]["40"] = "Fa0/20-24"
    print("\nValidation of a bad change request:", validate(bad))
    print("\nJSON is just text: first 80 characters of the inventory ->", json.dumps(INV)[:80] + "...")

Source of truth for Karachi-Campus: 3 VLANs, 2 switches
Validation: OK

Generated config for ACC-SW1:
  hostname ACC-SW1
  ntp server 192.168.99.1
  logging host 192.168.99.30
  vlan 10
   name LECTURERS
  vlan 20
   name STUDENTS
  vlan 99
   name MANAGEMENT
  interface range Fa0/1 - 12
   switchport mode access
   switchport access vlan 10
   switchport nonegotiate
   spanning-tree portfast
  interface range Fa0/13 - 24
   switchport mode access
   switchport access vlan 20
   switchport nonegotiate
   spanning-tree portfast
  interface vlan 99
   ip address 192.168.99.11 255.255.255.0

Drift check on ACC-SW2 (intended vs running):
  --- intended
  +++ running
  @@ -2 +2 @@
  -ntp server 192.168.99.1
  +ntp server 10.0.0.1
  @@ -5,0 +6 @@
  + name TEMP-TEST

Validation of a bad change request: ['192.168.20.0/23 overlaps 192.168.21.0/24', 'ACC-SW2 uses undefined VLAN 40']

JSON is just text: first 80 characters of the inventory -> {"site": "Karachi-Campus", "ntp": "192.168.99.1", "sys

## `session-15_rest_controller.py`

In [2]:
"""Session 15 - A REST API like a network controller's, running only on 127.0.0.1.

Starts a tiny mock controller with Python's http.server, then acts as a client:
- POST /api/v1/auth/token        -> get a token (like a controller login)
- GET  /api/v1/devices           -> inventory as JSON
- GET  /api/v1/devices/<name>    -> one device
- POST /api/v1/vlans             -> create a VLAN (validated, 201 Created)
- requests without a token get 401 Unauthorized
Nothing leaves your computer.
Run:  python session-15_rest_controller.py
"""
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, HTTPServer

TOKEN = "lab-token-123"
DEVICES = [{"hostname": "CORE-SW1", "role": "distribution", "ip": "192.168.99.2", "os": "IOS-XE 17.9", "reachable": True},
           {"hostname": "ACC-SW1", "role": "access", "ip": "192.168.99.11", "os": "IOS 15.2", "reachable": True},
           {"hostname": "EDGE-R1", "role": "border", "ip": "192.168.99.1", "os": "IOS-XE 17.6", "reachable": False}]
VLANS = {10: "LECTURERS", 20: "STUDENTS"}


class Controller(BaseHTTPRequestHandler):
    def log_message(self, *a):             # keep the output tidy
        pass

    def send(self, code, body):
        data = json.dumps(body).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers(); self.wfile.write(data)

    def authorised(self):
        return self.headers.get("X-Auth-Token") == TOKEN

    def do_GET(self):
        if not self.authorised():
            return self.send(401, {"error": "missing or invalid token"})
        if self.path == "/api/v1/devices":
            return self.send(200, {"response": DEVICES})
        if self.path.startswith("/api/v1/devices/"):
            name = self.path.rsplit("/", 1)[1]
            d = next((x for x in DEVICES if x["hostname"] == name), None)
            return self.send(200, {"response": d}) if d else self.send(404, {"error": "not found"})
        self.send(404, {"error": "no such endpoint"})

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
        if self.path == "/api/v1/auth/token":
            ok = body == {"username": "netops", "password": "lab-only"}
            return self.send(200, {"token": TOKEN}) if ok else self.send(401, {"error": "bad credentials"})
        if not self.authorised():
            return self.send(401, {"error": "missing or invalid token"})
        if self.path == "/api/v1/vlans":
            vid = body.get("id")
            if not isinstance(vid, int) or not 1 <= vid <= 4094:
                return self.send(400, {"error": "id must be 1-4094"})
            if vid in VLANS:
                return self.send(409, {"error": f"VLAN {vid} exists"})
            VLANS[vid] = body.get("name", f"VLAN{vid:04d}")
            return self.send(201, {"created": {"id": vid, "name": VLANS[vid]}})
        self.send(404, {"error": "no such endpoint"})


LOCAL = urllib.request.build_opener(urllib.request.ProxyHandler({}))   # never route 127.0.0.1 via a proxy


def call(method, url, body=None, token=None):
    req = urllib.request.Request(url, method=method, data=json.dumps(body).encode() if body is not None else None,
                                 headers={"Content-Type": "application/json", **({"X-Auth-Token": token} if token else {})})
    try:
        with LOCAL.open(req, timeout=5) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read())


if __name__ == "__main__":
    srv = HTTPServer(("127.0.0.1", 0), Controller)
    base = f"http://127.0.0.1:{srv.server_address[1]}/api/v1"
    threading.Thread(target=srv.serve_forever, daemon=True).start()

    print("GET /devices without a token  ->", call("GET", base + "/devices"))
    code, tok = call("POST", base + "/auth/token", {"username": "netops", "password": "lab-only"})
    token = tok["token"]
    print("POST /auth/token              ->", code, "token received")
    code, data = call("GET", base + "/devices", token=token)
    print(f"GET /devices                  -> {code}, {len(data['response'])} devices")
    for d in data["response"]:
        print(f"   {d['hostname']:9} {d['role']:12} {d['ip']:14} {d['os']:12} {'UP' if d['reachable'] else 'UNREACHABLE'}")
    print("GET /devices/ACC-SW1          ->", call("GET", base + "/devices/ACC-SW1", token=token))
    print("POST /vlans {id: 30, LAB}     ->", call("POST", base + "/vlans", {"id": 30, "name": "LAB"}, token))
    print("POST /vlans {id: 30} again    ->", call("POST", base + "/vlans", {"id": 30, "name": "LAB"}, token))
    print("POST /vlans {id: 5000}        ->", call("POST", base + "/vlans", {"id": 5000}, token))
    srv.shutdown()

GET /devices without a token  -> (401, {'error': 'missing or invalid token'})
POST /auth/token              -> 200 token received
GET /devices                  -> 200, 3 devices
   CORE-SW1  distribution 192.168.99.2   IOS-XE 17.9  UP
   ACC-SW1   access       192.168.99.11  IOS 15.2     UP
   EDGE-R1   border       192.168.99.1   IOS-XE 17.6  UNREACHABLE
GET /devices/ACC-SW1          -> (200, {'response': {'hostname': 'ACC-SW1', 'role': 'access', 'ip': '192.168.99.11', 'os': 'IOS 15.2', 'reachable': True}})
POST /vlans {id: 30, LAB}     -> (201, {'created': {'id': 30, 'name': 'LAB'}})
POST /vlans {id: 30} again    -> (409, {'error': 'VLAN 30 exists'})
POST /vlans {id: 5000}        -> (400, {'error': 'id must be 1-4094'})


## `mini_project_campus_builder.py`

In [3]:
"""Module 4 mini project - Campus network builder: from intent to device configs.

Input: what the campus needs (departments, host counts, which ones are guests).
Output, all generated and checked automatically:
  1. A VLSM address plan with VLAN IDs, gateways and DHCP ranges.
  2. Two distribution switches (DIST-1, DIST-2) with SVIs and HSRP, where the
     HSRP Active router and the STP root are the SAME switch for each VLAN
     (odd VLANs on DIST-1, even VLANs on DIST-2) so load is shared.
  3. DHCP pools, a guest-isolation ACL, and DHCP snooping/edge-port settings.
  4. A pre-deployment audit (overlaps, gateway inside subnet, sizing headroom).
Run:  python mini_project_campus_builder.py
"""
import ipaddress
import math

CAMPUS_BLOCK = "10.60.0.0/21"
DNS = None                # set by plan(): 4th address of the SERVERS subnet
INTENT = [  # name, expected hosts, guest network?
    ("LECTURERS", 120, False),
    ("STUDENT-LAB", 300, False),
    ("ADMIN", 40, False),
    ("GUEST-WIFI", 200, True),
    ("PRINTERS-IOT", 20, False),
    ("SERVERS", 12, False),
]
GROWTH = 1.25            # leave 25% room for growth


def plan(block, intent):
    free = [ipaddress.ip_network(block)]
    rows = []
    for i, (name, hosts, guest) in enumerate(sorted(intent, key=lambda x: -x[1])):
        need = math.ceil(hosts * GROWTH) + 3          # + gateway, DIST-1, DIST-2
        prefix = 32 - math.ceil(math.log2(need + 2))
        cand = sorted((f for f in free if f.prefixlen <= prefix), key=lambda n: int(n.network_address))
        if not cand:
            raise SystemExit(f"Address block too small for {name}")
        parent = cand[0]
        net = next(parent.subnets(new_prefix=prefix))
        free.remove(parent); free.extend(parent.address_exclude(net))
        rows.append({"name": name, "hosts": hosts, "guest": guest, "net": net})
    for vid, r in zip(range(10, 10 * (len(rows) + 1), 10), sorted(rows, key=lambda r: int(r["net"].network_address))):
        r["vlan"] = vid
        h = list(r["net"].hosts())
        r["vip"], r["d1"], r["d2"] = h[0], h[1], h[2]
        r["dhcp_from"], r["dhcp_to"] = h[3], h[-1]
        if r["name"] == "SERVERS":                    # first 10 server addresses stay static (DNS = first)
            global DNS
            DNS = h[3]
            r["dhcp_from"] = h[13]
    return sorted(rows, key=lambda r: r["vlan"]), sorted(free, key=lambda n: int(n.network_address))


def dist_config(rows, me):
    out = [f"hostname DIST-{me}", "ip routing", "spanning-tree mode rapid-pvst"]
    mine = [r["vlan"] for r in rows if (r["vlan"] // 10) % 2 == (1 if me == 1 else 0)]
    other = [r["vlan"] for r in rows if r["vlan"] not in mine]
    out.append(f"spanning-tree vlan {','.join(map(str, mine))} root primary")
    out.append(f"spanning-tree vlan {','.join(map(str, other))} root secondary")
    for r in rows:
        active = r["vlan"] in mine
        out += [f"vlan {r['vlan']}", f" name {r['name']}",
                f"interface Vlan{r['vlan']}",
                f" ip address {r['d1'] if me == 1 else r['d2']} {r['net'].netmask}",
                " standby version 2",
                f" standby {r['vlan']} ip {r['vip']}",
                f" standby {r['vlan']} priority {110 if active else 100}",
                f" standby {r['vlan']} preempt"]
        if r["guest"]:
            out.append(" ip access-group GUEST-IN in")
        out.append(" no shutdown")
    return out


def services(rows):
    out = []
    for r in rows:
        out += [f"ip dhcp excluded-address {r['net'].network_address + 1} {r['dhcp_from'] - 1}",
                f"ip dhcp pool {r['name']}", f" network {r['net'].network_address} {r['net'].netmask}",
                f" default-router {r['vip']}", f" dns-server {DNS}"]
    internal = [r for r in rows if not r["guest"]]
    guest = next(r for r in rows if r["guest"])
    gw = guest["net"].hostmask
    out += ["ip access-list extended GUEST-IN",
            f" permit udp {guest['net'].network_address} {gw} host {DNS} eq 53"]
    out += [f" deny ip {guest['net'].network_address} {gw} {r['net'].network_address} {r['net'].hostmask}" for r in internal]
    out += [f" permit ip {guest['net'].network_address} {gw} any",
            "! access switches", f"ip dhcp snooping vlan {','.join(str(r['vlan']) for r in rows)}",
            "interface range Gi1/0/1 - 46", " switchport mode access", " switchport nonegotiate",
            " spanning-tree portfast", " spanning-tree bpduguard enable", " ip dhcp snooping limit rate 15",
            "interface range Gi1/0/47 - 48", " ip dhcp snooping trust"]
    return out


def audit(rows):
    issues = []
    nets = [r["net"] for r in rows]
    for i, a in enumerate(nets):
        issues += [f"overlap {a} {b}" for b in nets[i + 1:] if a.overlaps(b)]
    for r in rows:
        if r["vip"] not in r["net"]:
            issues.append(f"{r['name']}: gateway outside subnet")
        usable = r["net"].num_addresses - 2 - 3
        if usable < r["hosts"] * GROWTH:
            issues.append(f"{r['name']}: not enough room for growth")
    return issues


if __name__ == "__main__":
    rows, free = plan(CAMPUS_BLOCK, INTENT)
    print(f"Campus block {CAMPUS_BLOCK}  (growth allowance {int((GROWTH - 1) * 100)}%)\n")
    print(f"{'VLAN':>4}  {'Name':13} {'Hosts':>5}  {'Subnet':16} {'Gateway (VIP)':14} {'DHCP range':29} Active")
    for r in rows:
        act = "DIST-1" if (r["vlan"] // 10) % 2 == 1 else "DIST-2"
        print(f"{r['vlan']:>4}  {r['name']:13} {r['hosts']:>5}  {str(r['net']):16} {str(r['vip']):14} "
              f"{str(r['dhcp_from']) + ' - ' + str(r['dhcp_to']):29} {act}")
    print("Spare space:", ", ".join(map(str, free)) or "none")
    print("\nAudit:", audit(rows) or "PASS - no overlaps, gateways valid, growth room OK")
    d1 = dist_config(rows, 1)
    print(f"\nDIST-1 configuration ({len(d1)} lines), first VLAN shown:")
    print("\n".join("  " + l for l in d1[:13]))
    svc = services(rows)
    print(f"\nShared services ({len(svc)} lines), guest ACL shown:")
    start = svc.index("ip access-list extended GUEST-IN")
    print("\n".join("  " + l for l in svc[start:start + 8]))

Campus block 10.60.0.0/21  (growth allowance 25%)

VLAN  Name          Hosts  Subnet           Gateway (VIP)  DHCP range                    Active
  10  STUDENT-LAB     300  10.60.0.0/23     10.60.0.1      10.60.0.4 - 10.60.1.254       DIST-1
  20  GUEST-WIFI      200  10.60.2.0/24     10.60.2.1      10.60.2.4 - 10.60.2.254       DIST-2
  30  LECTURERS       120  10.60.3.0/24     10.60.3.1      10.60.3.4 - 10.60.3.254       DIST-1
  40  ADMIN            40  10.60.4.0/26     10.60.4.1      10.60.4.4 - 10.60.4.62        DIST-2
  50  PRINTERS-IOT     20  10.60.4.64/27    10.60.4.65     10.60.4.68 - 10.60.4.94       DIST-1
  60  SERVERS          12  10.60.4.96/27    10.60.4.97     10.60.4.110 - 10.60.4.126     DIST-2
Spare space: 10.60.4.128/25, 10.60.5.0/24, 10.60.6.0/23

Audit: PASS - no overlaps, gateways valid, growth room OK

DIST-1 configuration (60 lines), first VLAN shown:
  hostname DIST-1
  ip routing
  spanning-tree mode rapid-pvst
  spanning-tree vlan 10,30,50 root primary
  sp